# TASK OVERVIEW

This project evaluates LLMs on preference-based matching problems using 4 tasks:

1. Task 1: Generate a stable matching  
2. Task 2: Detect instability (is matching stable or not)  
3. Task 3: Resolve instability (fix an unstable matching)  
4. Task 4: Preference reasoning (queries over rankings)

Models used:
- Basic model (baseline)
- Reasoning model (expected to perform better)

Dataset:
- Small instances (n = 5) using IC preferences

=========================
COMMON SETUP AND HELPERS
=========================
Goal: Define shared functions and utilities used across all tasks.

Includes:

- dataset loading
- preference parsing
- instance preparation
- answer template generation
- shared utility functions

In [1]:
!pip install -q langchain-groq

In [2]:
!pip install together

In [20]:
import os
import re
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import sys
from getpass import getpass
from langchain_groq import ChatGroq

In [21]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [24]:
import sys
from pathlib import Path

# Current working directory is likely .../project/notebooks
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Add the actual project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [29]:
from src.config import setup_api_key, DATA_DIR, RESULTS_DIR
from src.config import setup_api_key, DATA_DIR, RESULTS_DIR
from src.task1_stable_matching import basic_model, reasoning_model
from src.task2_detect_instability import task2_basic_model, task2_reasoning_model
from src.task3_resolve_instability import task3_basic_model, task3_reasoning_model
from src.task4_preference_reasoning import task4_basic_model, task4_reasoning_model
# from src.reporting import (
#     build_final_summary_table,
#     build_chart_dataframe,
#     plot_grouped_bar_chart,
#     save_dataframe,
# )

In [30]:
GROQ_API_KEY = getpass("Enter GROQ API key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CSV_FILE = DATA_DIR / "5_ic_processed.csv"

# =========================
# TASK 1: GENERATE STABLE MATCHING
# =========================


Goal:
Generate a stable matching from given preference lists.

Input:
- Preference lists for men and women

Output:
- One-to-one matching in JSON format

Evaluation:
- Parsed correctly
- Valid matching (one-to-one)
- Stable (no blocking pairs)
- Exact match with ground truth
- Blocking pairs count

In [31]:
basic_detailed, basic_results, basic_summary = basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)


parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 4
blocking pairs: [('M1', 'W3'), ('M2', 'W3'), ('M3', 'W3'), ('M4', 'W3')]

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 1
blocking pairs: [('M1', 'W3')]

{'total_instances': 10, 'parsed_count': 10, 'valid_count': 10, 'stable_count': 3, 'exact_match_count': 1, 'avg_blocking_pairs': 2.0}


In [32]:
reasoning_detailed, reasoning_results, reasoning_summary = reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=25,
    num_examples_to_show=2
)

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 3
blocking pairs: [('M4', 'W2'), ('M4', 'W5'), ('M5', 'W2')]

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 2
blocking pairs: [('M3', 'W5'), ('M3', 'W1')]

{'total_instances': 25, 'parsed_count': 25, 'valid_count': 25, 'stable_count': 5, 'exact_match_count': 2, 'avg_blocking_pairs': 2.0}


### Task 1 Insights: Generating Stable Matchings

Both models were able to consistently produce valid one-to-one matchings, indicating that they understood the basic structure of the problem.

However, generating *stable* matchings proved challenging. The basic model produced only a small number of stable outcomes, while the reasoning model performed better but still failed to consistently eliminate blocking pairs.

A key observation is that most outputs were close to correct, often containing only a small number of blocking pairs. This suggests that while the models partially capture the structure of the problem, they struggle to enforce global stability constraints.

Overall, the reasoning model outperformed the basic model, but neither model reliably generated stable matchings even for small instances (n = 5).

# =========================
# TASK 2: DETECT INSTABILITY
# =========================

Goal:
Given a matching, determine whether it is stable or unstable.

Output:
- YES (stable) or NO (unstable)

In [38]:
# =========================================
# 2.7 run task 2 experiments
# =========================================

task2_basic_detailed, task2_basic_summary = task2_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 2 basic summary")
print(task2_basic_summary)

print("\n" + "="*50 + "\n")

task2_reasoning_detailed, task2_reasoning_summary = task2_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 2 reasoning summary")
print(task2_reasoning_summary)

NameError: name 'ChatGroq' is not defined

### Task 2 Insights: Detecting Instability

In this task, both models were evaluated on their ability to classify matchings as stable or unstable.

The basic model achieved an accuracy of 50%, indicating near-random performance. In contrast, the reasoning model achieved 70% accuracy, demonstrating a clear improvement in its ability to analyze stability conditions.

A notable pattern is that both models frequently predicted matchings as unstable, even when they were stable. This suggests a bias toward detecting instability, possibly due to difficulty verifying the absence of blocking pairs.

Overall, instability detection is easier than generating stable matchings, but still non-trivial. The reasoning model shows improved performance, yet still makes systematic errors.

=========================
TASK 3: RESOLVE INSTABILITY
=========================
Goal: Given an unstable matching, modify it to produce a stable matching.

Output:

A valid one-to-one stable matching in JSON format

In [ ]:
# =========================================
# 3.5 run task 3 experiments
# =========================================

task3_basic_detailed, task3_basic_summary = task3_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 3 basic summary")
print(task3_basic_summary)

print("\n" + "=" * 50 + "\n")

task3_reasoning_detailed, task3_reasoning_summary = task3_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 3 reasoning summary")
print(task3_reasoning_summary)

In [ ]:
# =========================================
# 3.6 task 3 quick comparison
# =========================================

print("task 3 comparison")
print(f"basic stable count: {task3_basic_summary['stable_count']} / {task3_basic_summary['total_instances']}")
print(f"reasoning stable count: {task3_reasoning_summary['stable_count']} / {task3_reasoning_summary['total_instances']}")
print(f"basic exact match count: {task3_basic_summary['exact_match_count']} / {task3_basic_summary['total_instances']}")
print(f"reasoning exact match count: {task3_reasoning_summary['exact_match_count']} / {task3_reasoning_summary['total_instances']}")
print(f"basic avg blocking pairs: {task3_basic_summary['avg_blocking_pairs']}")
print(f"reasoning avg blocking pairs: {task3_reasoning_summary['avg_blocking_pairs']}")

### Task 3 Insights: Resolving Instability

This task required the models to transform an unstable matching into a stable one.

Both models struggled significantly. The basic model produced only 1 stable matching out of 10 instances, while the reasoning model improved slightly to 3 out of 10.

Although most outputs were valid matchings, they often remained unstable, with an average of approximately 1–2 blocking pairs. This indicates that the models were unable to effectively eliminate all instability, even when explicitly instructed to do so.

Compared to Task 2, where models could sometimes recognize instability, this task highlights that *correcting* instability is substantially more difficult than detecting it.

Overall, resolving instability requires deeper algorithmic reasoning, and both models show clear limitations in this setting.

=========================
TASK 4: PREFERENCE REASONING
=========================
Goal: Answer questions based on preference lists of agents.

Output:

- Agent name (for ranking questions)
- YES or NO (for comparison and decision questions)

In [34]:
# =========================================
# 4.7 run task 4 experiments
# =========================================

task4_basic_detailed, task4_basic_summary = task4_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 4 basic summary")
print(task4_basic_summary)

print("\n" + "=" * 50 + "\n")

task4_reasoning_detailed, task4_reasoning_summary = task4_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 4 reasoning summary")
print(task4_reasoning_summary)

NameError: name 'ChatGroq' is not defined

In [ ]:
# =========================================
# 4.8 task 4 quick comparison
# =========================================

print("task 4 comparison")
print(f"basic correct count: {task4_basic_summary['correct_count']} / {task4_basic_summary['total_instances']}")
print(f"reasoning correct count: {task4_reasoning_summary['correct_count']} / {task4_reasoning_summary['total_instances']}")
print(f"basic accuracy: {task4_basic_summary['accuracy']}")
print(f"reasoning accuracy: {task4_reasoning_summary['accuracy']}")

### Task 4 Insights: Preference Reasoning

Both models performed better on preference reasoning compared to other tasks. The basic model achieved 60% accuracy, while the reasoning model achieved 70%.

Unlike Tasks 1 and 3, which require enforcing global stability constraints, this task focuses on local reasoning over preference lists. The higher accuracy indicates that LLMs are more effective at answering direct queries about rankings and comparisons.

However, errors still occur, particularly in yes/no comparison questions, suggesting that even local reasoning is not fully reliable.

Overall, this task highlights that LLMs are stronger at interpreting preferences than constructing or correcting stable matchings.